# Extract Glassdoor Company ID

code modified from https://scrapfly.io/blog/how-to-scrape-glassdoor/

### API and Scrape function
find_companies is used to find companies' unique Glassdoor ID by scraping Glassdoor search suggestions with Scrapfly API

To save Scrapfly API quota, the cost budget is set to 5 which means the scraping bot only try scrap once. The failed task will be tried again manually later.

In [ ]:
import json
from scrapfly import ScrapeApiResponse, ScrapeConfig, ScrapflyClient
from urllib.parse import urlencode 

client = ScrapflyClient(key="   ")

def find_companies(query: str):
    """find company Glassdoor ID and name by query. e.g. "ebay" will return "eBay" with ID 7853"""

    prams = {
        "numSuggestions": 8,
        "source": "GD_V2",
        "version": "NEW",
        "rf": "full",
        "fallback": "token",
        "input": query
    }
    url=f"https://www.glassdoor.com/searchsuggest/typeahead?" + urlencode(prams)
    result = client.scrape(
        ScrapeConfig(
            url=url,
            country="US",
            asp=True,
            cookies={"tldp":"1"},  # sets location to US
            cost_budget=5
        )
    )
    if result.status_code != 200:
        return 0
    data = json.loads(result.content)
    if not data or len(data) == 0:
        return 0
    return data[0]["employerId"]


Find_companies_pure tried to scrape without using Scrape API. However, it will be blocked by anti-scraping in Glassdoor. So this function was not obseleted.

Note: tried many customized header and cookies, all were blocked

In [2]:
import httpx

# find company Glassdoor ID with no scrape api call
def find_companies_pure(query: str):
    url = f"https://www.glassdoor.com/searchsuggest/typeahead"
    prams = {
        "numSuggestions": 8,
        "source": "GD_V2",
        "version": "NEW",
        "rf": "full",
        "fallback": "token",
        "input": query
    }
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3",
        "cookie": 'gdId=1edda9bf-930b-432e-8f61-fd1c051b7ff7; rl_page_init_referrer=RudderEncrypt%3AU2FsdGVkX1%2BGD0wn3jPsvUM3YkwHv7ya2u1LzBzCnCTYKNokj8%2BCgJJW2zLc0O1v; rl_page_init_referring_domain=RudderEncrypt%3AU2FsdGVkX1%2BccEUHfdn%2BoKPpXYgISoK1usAHYRAsrIwjigwR6A3yErGfCs2Pqzu8; _ga=GA1.2.1744269303.1710448413; _optionalConsent=true; _gcl_au=1.1.231913175.1710448413; __pdst=d8d6ffd1b6344c48a5473e9f8c1eb8ef; _tt_enable_cookie=1; _ttp=z25_k5sjg-5dxKSMV8VtipzAQs1; _pin_unauth=dWlkPU9USmhZMlJpTTJZdFlXTTNNQzAwWWpjM0xXRmhOekV0WVRVNE9HSTJZalk1WldGaQ; g_state={"i_p":1712011183313,"i_l":1}; trs=https%3A%2F%2Fwww.reddit.com%2F:referral:referral:2024-04-01+14%3A04%3A29.244:undefined:undefined; optimizelyEndUserId=oeu1712005893379r0.07747710842660416; ki_u=6f219cf3-3f6e-12af-931b-b47c; uc=8013A8318C98C5173AB185891022C2AEF9C6942DF7FE053F14CF16596848AA65D0EB5FB416099D626166E0EDBB1E343C8236C800F4C00551AA033FC3DDC61F388BF22F3F1172E572AA28148B867637B113CF0B5F6B0083B7E3E58C98EFC1F795A2D7A75A9DE91C0B18B8E2EFA492A88D453B6FCD835247D5A4A094EE94F193BEE93A7ACD799185BFED894F1FA40A7891; ki_r=aHR0cHM6Ly93d3cuZ29vZ2xlLmNvbS8%3D; alr=https%3A%2F%2Fwww.google.com%2F; _cfuvid=iAZ8QNAeH2s50fy1JeQqJE7sliJVVuXdNx_OfkmwbiE-1713118128398-0.0.1.1-604800000; _gid=GA1.2.177916647.1713118130; GSESSIONID=undefined; rsSessionId=1713118129723; JSESSIONID=012A4760899438CCF73E781D9DBB901E; cass=1; ki_s=200550%3A0.0.0.0.0%3B213982%3A0.0.0.0.0%3B235378%3A0.0.0.0.0%3B235518%3A0.0.0.0.1%3B237382%3A0.0.0.0.0; cf_chl_3=04195eccc46d430; _ga_500H17GZ8D=GS1.2.1713120790.1.0.1713120790.60.0.0; cf_clearance=B8ROPgQOhEikwc6Sxfv8xOie7Hr75RDFQA_6BjVPBPM-1713120789-1.0.1.1-TfzSmVTirmtlNvwn4RE3kAp1TLDr03Qmy1SxlkWAijRfx49Vb2Y7LcLFSe3NFz8Fp_VwySOhSZrP_81FWXSfJA; indeedCtk=1hrev9mkaj4qo801; gdsid=1713118127821:1713122563978:7EA7450F7C438D09614F25818203D17F; asst=1713122563.2; rl_user_id=RudderEncrypt%3AU2FsdGVkX1%2BrXL11sQFWVn2gZZuu4qxkeLPsEhuJrbQ%3D; rl_trait=RudderEncrypt%3AU2FsdGVkX1%2FYyQJTlp7SNZhFt7FwIaFnBIcvIDvLmm%2FRYaWWUFoksQnP5thbGy2x83OU4AgqC98G8dgnOe6KeDxR2VISP9LLf8x0GtIwsn24RZ0RCgT8Jj9gWpDLDafifMR0jWVZE0Ae2f7uiTbnhjtLuRUA4KMs3ihIdkCO7RWFwb4SmA6cdE3LTRksLGBalGZm7S%2F28nEFTCrmRL1rE0WdccBmHKdpnMLwQkqzvho%3D; rl_group_id=RudderEncrypt%3AU2FsdGVkX19wFMcCRkvMg9JUYSoT7J%2FDDOOy3FtEH2Q%3D; rl_group_trait=RudderEncrypt%3AU2FsdGVkX19U4BhIL%2FsKF7pVwctJe%2FwOTsfy57s6QkU%3D; rl_anonymous_id=RudderEncrypt%3AU2FsdGVkX19w51Dbp7EVjLZ3EQwE46zry3KwMkiOdiiYkhaJf38EjR1eKZiDf0SFCw3%2BihstMd8ZGt%2FrWu2J%2Fg%3D%3D; rsReferrerData=%7B%22currentPageRollup%22%3A%22%2Findex%22%2C%22previousPageRollup%22%3A%22%2Foverview%2Fworking-at%22%2C%22currentPageAbstract%22%3A%22%2Findex.htm%22%2C%22previousPageAbstract%22%3A%22%2FOverview%2FWorking-at-%5BEMP%5D-EI_%5BPRM%5D.htm%22%2C%22currentPageFull%22%3A%22https%3A%2F%2Fwww.glassdoor.com%2Findex.htm%22%2C%22previousPageFull%22%3A%22https%3A%2F%2Fwww.glassdoor.com%2FOverview%2FWorking-at-AT-and-T-EI_IE613.11%2C19.htm%22%7D; AWSALB=cJ2SXXPPW9lwjOSxuXgDUBUn3EC3HogrrOa0oPd5iR8IzEZXMkeMIhgq6QHNAOeiTLZUI/iy9lcNlrXqWEQypKfQMMW0R9u0r0gWtNVWutWbSqsSLwIVx36qTR+Q; AWSALBCORS=cJ2SXXPPW9lwjOSxuXgDUBUn3EC3HogrrOa0oPd5iR8IzEZXMkeMIhgq6QHNAOeiTLZUI/iy9lcNlrXqWEQypKfQMMW0R9u0r0gWtNVWutWbSqsSLwIVx36qTR+Q; rl_session=RudderEncrypt%3AU2FsdGVkX18SL3GkmEuiXSDmPiv09X00miFgkhuha7jy87u2CQlP7QO19xn0EGm1iVdR9Ln%2FHtxravl7vvy8qCdu46ph9yWOXS9tYdwZIoH7%2FUSEoX2xwpajKPq3mXB19BvE2lr7RdPwI8ETwr7CXQ%3D%3D; OptanonConsent=isGpcEnabled=0&datestamp=Sun+Apr+14+2024+15%3A22%3A50+GMT-0400+(Eastern+Daylight+Time)&version=202403.2.0&browserGpcFlag=0&isIABGlobal=false&hosts=&consentId=ada24524-eefe-4d79-8e4f-71662e8f3220&interactionCount=1&landingPath=NotLandingPage&groups=C0001%3A1%2CC0003%3A1%2CC0002%3A1%2CC0004%3A1%2CC0017%3A1&AwaitingReconsent=false&isAnonUser=1; _rdt_uuid=1710448413374.542bdba1-9dc1-40cc-84bf-31dcbaf3ea1c; ki_t=1710448413816%3B1713118188565%3B1713122571244%3B4%3B29; __cf_bm=N3C4Qw81zT4rVwYNBbLsEDJbVK.dpb3kI2d0J2Wtw5g-1713122589-1.0.1.1-X95Tc9sfsd3FFRdFKpeS7KcTUg7JTyTLITLVWFVM.sTj0YnCkmzvS3742PBJikLGigsVwDYy04eV50INdqiE_elUmJlrXvbjKwvidbSI7Q4'
    }
    response = httpx.get(url, params=prams, headers=headers)
    
    print(response)
    return 0
    # if not data or len(data) == 0:
    #     return None
    
    # data = json.loads(response.content)
    # return data[0]["employerId"]


### Read Company List from top500.json


In [19]:
# read from a json file
with open("top500_id.json") as f:
    companies = json.load(f)

print(companies[:3])

[{'rank': '1', 'company': 'Walmart', 'industry': 'General Merchandisers', 'city': 'Bentonville', 'state': 'Arkansas', 'zip': '72716', 'website': 'www.walmart.com', 'employees': '2,100,000', 'revenue': '$611,289', 'valuation': '$397,475', 'profit': '$11,680', 'ticker': 'WMT', 'ceo': 'Douglas McMillon', 'employerId': '715'}, {'rank': '2', 'company': 'Amazon.com', 'industry': 'Internet Services and Retailing', 'city': 'Seattle', 'state': 'Washington', 'zip': '98109', 'website': 'www.amazon.com', 'employees': '1,541,000', 'revenue': '$513,983', 'valuation': '$1,058,440', 'profit': '-$2,722', 'ticker': 'AMZN', 'ceo': 'Andrew R. Jassy', 'employerId': '6036'}, {'rank': '3', 'company': 'Exxon Mobil', 'industry': 'Petroleum Refining', 'city': 'Irving', 'state': 'Texas', 'zip': '75039', 'website': 'www.exxonmobil.com', 'employees': '62,000', 'revenue': '$413,680', 'valuation': '$446,424', 'profit': '$55,740', 'ticker': 'XOM', 'ceo': 'Darren W. Woods', 'employerId': '237'}]


### Call Scrape API for each company and retrieve its Glassdoor ID

In [20]:

not_found = {}
log = []

for i in range(len(companies)):
    try:
        if 'employerId' not in companies[i]:
            employerId = find_companies(companies[i]['company'])
            companies[i]['employerId'] = employerId
            
            if employerId is None or employerId == 0:
                not_found[companies[i]['company']] = companies[i]
    except Exception as e:
        not_found[companies[i]['company']] = companies[i]
        # print(e)
        log.append((repr(e), companies[i]))
    
if len(log) > 0:
    with open("top500_log.txt", "w") as f:
        for e in log:
            f.write("========================================\n")
            f.write(f"{e[0]}\n")
            f.write("----------------------------------------\n")
            f.write(f"{e[1]}\n")
            f.write("\n")


print(companies[:3])
print( "number of company ids not found: {}".format(len(not_found)))

CRITICAL:root:<-- 422 | ERR::SCRAPE::COST_BUDGET_LIMIT - Cost budget has been reached, you must increase the budget to pass this target - The cost of the scrape is exceeding the budget of 5, would require 51 API credit.Checkout the related doc: https://scrapfly.io/docs/scrape-api/anti-scraping-protection
CRITICAL:root:<-- 422 | ERR::SCRAPE::COST_BUDGET_LIMIT - Cost budget has been reached, you must increase the budget to pass this target - The cost of the scrape is exceeding the budget of 5, would require 51 API credit.Checkout the related doc: https://scrapfly.io/docs/scrape-api/anti-scraping-protection
CRITICAL:root:<-- 422 | ERR::SCRAPE::COST_BUDGET_LIMIT - Cost budget has been reached, you must increase the budget to pass this target - The cost of the scrape is exceeding the budget of 5, would require 51 API credit.Checkout the related doc: https://scrapfly.io/docs/scrape-api/anti-scraping-protection
CRITICAL:root:<-- 422 | ERR::SCRAPE::COST_BUDGET_LIMIT - Cost budget has been reac

[{'rank': '1', 'company': 'Walmart', 'industry': 'General Merchandisers', 'city': 'Bentonville', 'state': 'Arkansas', 'zip': '72716', 'website': 'www.walmart.com', 'employees': '2,100,000', 'revenue': '$611,289', 'valuation': '$397,475', 'profit': '$11,680', 'ticker': 'WMT', 'ceo': 'Douglas McMillon', 'employerId': '715'}, {'rank': '2', 'company': 'Amazon.com', 'industry': 'Internet Services and Retailing', 'city': 'Seattle', 'state': 'Washington', 'zip': '98109', 'website': 'www.amazon.com', 'employees': '1,541,000', 'revenue': '$513,983', 'valuation': '$1,058,440', 'profit': '-$2,722', 'ticker': 'AMZN', 'ceo': 'Andrew R. Jassy', 'employerId': '6036'}, {'rank': '3', 'company': 'Exxon Mobil', 'industry': 'Petroleum Refining', 'city': 'Irving', 'state': 'Texas', 'zip': '75039', 'website': 'www.exxonmobil.com', 'employees': '62,000', 'revenue': '$413,680', 'valuation': '$446,424', 'profit': '$55,740', 'ticker': 'XOM', 'ceo': 'Darren W. Woods', 'employerId': '237'}]
number of company ids 

### Save companies without ID to non_found.json
If there is any company failed, save them to a json file and manually visit the url to find the ID

In [21]:
# save not found companies to a json file for manual processing
if len(not_found.items()) > 0:
    with open("not_found.json", "w") as f:
        json.dump(not_found, f)


**before next step, manual process the not_found.json (not in this code)**

### combine manually added data to companies data

In [22]:
if len(not_found.items()) > 0:
    with open("not_found.json") as f:
        not_found = json.load(f)
    for i in range(len(companies)):
        if companies[i]['company'] in not_found:
            companies[i]['employerId'] = not_found[companies[i]['company']]["employerId"]

### Save to top500_id.json

In [23]:
#save data to top500_id json file
with open("top500_id.json", "w") as f:
    json.dump(companies, f, indent=2)


